# L10n QA Orchestrator · Demostración ejecutable

Este notebook **ejecuta de verdad** el grafo multiagente de LangGraph sobre los fixtures del repositorio.
El sistema hace **QA estructural y validación de preparación para importación** de JSON traducidos
(por ejemplo, antes de importarlos a Sitecore o Umbraco). No hace QA lingüístico ni modifica archivos.

Flujo: `Supervisor → Investigación / Análisis → Supervisor → Validación → Fin`.
Los agentes comparten un estado (`MessagesState` + campos propios) y el Supervisor decide cada ruta en función de ese estado.

In [1]:
import json, sys
from pathlib import Path

RAIZ = Path.cwd().parent            # el notebook vive en notebooks/
sys.path.insert(0, str(RAIZ))

from IPython.display import Markdown, display
from app.graph import graph, obtener_mermaid
from app.cli import SOLICITUD_EJEMPLO
from app.report import informe_a_markdown
from app.state import ESTADO_HALLAZGO_ES, crear_estado_inicial

ORIGEN, DESTINO = str(RAIZ / "fixtures" / "source"), str(RAIZ / "fixtures" / "targets")
print("Repositorio:", RAIZ.name)

Repositorio: l10n-qa-orchestrator


## 1. Solicitud inicial

In [2]:
print(SOLICITUD_EJEMPLO)

Compará los archivos JSON de origen en inglés con sus versiones traducidas al español y al árabe. Investigá las reglas estructurales aplicables, identificá claves faltantes o inesperadas, diferencias de tipos de datos, problemas con placeholders y discrepancias en los saltos de línea. Determiná si los archivos cumplen los controles estructurales configurados para una posible importación al CMS y generá un informe de QA con evidencias.


## 2. Ejecución del grafo paso a paso

Usamos `graph.stream(..., stream_mode="updates")` para ver **cada ejecución de nodo en el orden real**.
Cada línea es un nodo que se ejecutó; el Supervisor aparece entre cada especialista porque estos siempre le devuelven el control.

In [3]:
estado = crear_estado_inicial(SOLICITUD_EJEMPLO, [ORIGEN], [DESTINO])   # max_steps = 12 por defecto
final = None
for modo, datos in graph.stream(estado, stream_mode=["updates", "values"], config={"recursion_limit": 50}):
    if modo == "updates":
        for nodo, upd in datos.items():
            extra = f" → próxima ruta: {upd['next_agent']}" if nodo == "supervisor" else ""
            print(f"  ▶ nodo ejecutado: {nodo:<11}{extra}")
    else:
        final = datos
print(f"\nPasos ejecutados: {final['step_count']} de {final['max_steps']}")

  ▶ nodo ejecutado: supervisor  → próxima ruta: research
  ▶ nodo ejecutado: research   
  ▶ nodo ejecutado: supervisor  → próxima ruta: research
  ▶ nodo ejecutado: research   
  ▶ nodo ejecutado: supervisor  → próxima ruta: analysis
  ▶ nodo ejecutado: analysis   
  ▶ nodo ejecutado: supervisor  → próxima ruta: validation
  ▶ nodo ejecutado: validation 

Pasos ejecutados: 8 de 12


## 3. Decisiones de enrutamiento del Supervisor

In [4]:
for d in final["routing_log"]:
    marca = "  ← refinamiento concreto" if d["refinamiento"] else ""
    print(f"Paso {d['paso']:>2} · {d['ruta']:<10}{marca}\n          {d['motivo']}")
print()
for r in final["refinement_log"]:
    print("Solicitud de refinamiento:", r["destino"], "/", r["tipo"], "→", r["detalle"])

Paso  1 · research  
          Falta evidencia: aún no se inspeccionaron los archivos ni se consultaron reglas.
Paso  3 · research    ← refinamiento concreto
          Las reglas recuperadas no cubren las categorías: tipos_datos, placeholders, metadatos, excepciones.
Paso  5 · analysis  
          La investigación es suficiente y falta el análisis estructural.
Paso  7 · validation
          Investigación y análisis suficientes: se pasa a la validación final.

Solicitud de refinamiento: research / buscar_reglas → {'categorias': ['tipos_datos', 'placeholders', 'metadatos', 'excepciones']}


## 4. Contribución del Agente de Investigación

Inspecciona los archivos con `inspect_localization_files`, consulta la base de conocimiento con `search_l10n_qa_rules`
y determina las excepciones aplicables. **No** hace el análisis estructural.

In [5]:
for c in [c for c in final["agent_contributions"] if c["agente"] == "research"]:
    print(f"Paso {c['paso']} [{c['tipo_analisis']}] {c['resumen']}")
    print("   evidencias:", c["evidencias"])
reglas = {}
for e in final["research_findings"]:
    if e["tipo"] == "reglas":
        reglas.update({r["id"]: r["titulo"] for r in e["datos"]["reglas"]})
print(f"\nReglas recuperadas ({len(reglas)}):")
for rid, titulo in sorted(reglas.items()):
    print("  -", rid, titulo)
exc = [x for e in final["research_findings"] if e["tipo"] == "excepciones" for x in e["datos"]["aplicables"]]
print("\nExcepciones aplicables:", [(x["id"], x["clave"]) for x in exc])

Paso 2 [investigacion_inicial] Inspeccioné 5 archivo(s) y 3 par(es) (destino: ar, es); recuperé 5 regla(s) y 2 excepción(es) aplicable(s).
   evidencias: ['inspect_localization_files', 'inferir_controles', 'search_l10n_qa_rules', 'excepciones_aplicables']
Paso 4 [refinamiento_reglas] Atendí el refinamiento: recuperé 8 regla(s) para tipos_datos, placeholders, metadatos, excepciones.
   evidencias: ['search_l10n_qa_rules']

Reglas recuperadas (13):
  - REG-CLV-001 Toda clave del origen debe existir en el destino
  - REG-CLV-002 No debe haber claves inesperadas en el destino
  - REG-DIF-001 Un texto igual al origen no es un error por sí mismo
  - REG-EXC-001 Excepción configurable: _gpiInternalData-toolName
  - REG-EXC-002 Excepción configurable: _gpiInternalData-rootPath
  - REG-EXC-003 Las excepciones se aplican por clave explícita y se documentan
  - REG-MET-001 Los metadatos _gpiInternalData-* deben coincidir con el origen
  - REG-NL-001 No se intercambian saltos de línea reales y sec

## 5. Hallazgos del Agente de Análisis

Comparación estructural determinista con evidencia verificable. Estados: `confirmado` (error o información),
`pendiente` (advertencia que requiere revisión) o `rechazado` (lo decide la validación).

In [6]:
for h in final["analysis_findings"]:
    print(f"{h['id']} [{h['severidad']:<7}] {Path(h['archivo']).name:<13} {h['ruta_json']:<28} {h['categoria']:<30} {ESTADO_HALLAZGO_ES[h['estado']]}")
print("\nEjemplo de evidencia (placeholder alterado):")
ej = next(h for h in final["analysis_findings"] if h["categoria"] == "placeholder_alterado")
print(json.dumps(ej["evidencia"], ensure_ascii=False, indent=2))

F-001 [info   ] clean.es.json _gpiInternalData-rootPath    metadato_diferencia_permitida  confirmado
F-002 [info   ] clean.es.json $                            diferencia_legitima            confirmado
F-003 [warning] home.ar.json  page.links[1].label          valor_vacio                    pendiente
F-004 [warning] home.ar.json  footer.help                  salto_linea_cantidad_distinta  pendiente
F-005 [info   ] home.ar.json  _gpiInternalData-toolName    metadato_diferencia_permitida  confirmado
F-006 [info   ] home.ar.json  _gpiInternalData-rootPath    metadato_diferencia_permitida  confirmado
F-007 [error  ] home.ar.json  _gpiInternalData-itemId      metadato_distinto              confirmado
F-008 [info   ] home.ar.json  $                            diferencia_legitima            confirmado
F-009 [error  ] home.es.json  page.description             clave_faltante                 confirmado
F-010 [error  ] home.es.json  page.promoBadge              clave_inesperada               con

## 6. Validación final

In [7]:
for v in final["validation_results"]["verificaciones"]:
    print(f"[{'OK' if v['ok'] else 'FALLA'}] {v['descripcion']}\n        {v['detalle']}")

[OK] Están presentes las contribuciones necesarias (investigación y análisis)
        Investigación, análisis y supervisión contribuyeron.
[OK] Los hallazgos tienen una estructura válida
        15 hallazgo(s) con estructura válida.
[OK] Cada hallazgo tiene evidencia suficiente para reproducirlo
        Todas las evidencias se reprodujeron releyendo los archivos.
[OK] Las traducciones y archivos originales permanecen intactos
        5 archivo(s) con huella SHA-256 idéntica antes y después.
[OK] No se superó el límite de ejecución
        Ejecución 8 de 12 permitidas.
[OK] El informe refleja fielmente los hallazgos
        El informe refleja fielmente los hallazgos producidos.


## 7. Informe de QA

In [8]:
informe = final["final_report"]
print(f"Estado general: {informe['estado_general']} ({informe['estado_general_es']})")
print(informe["descripcion_estado"])
print("Resumen:", informe["resumen"])
display(Markdown(informe_a_markdown(informe)))

Estado general: FAIL (FALLIDO)
Se detectaron errores estructurales confirmados que impedirían o comprometerían la importación.
Resumen: {'errores': 6, 'advertencias': 2, 'informacion': 7, 'total': 15}


# Informe de QA estructural de localización

> Este sistema realiza exclusivamente QA estructural y validación de preparación para importación. No realiza QA lingüístico, no corrige ni completa traducciones y no modifica los archivos originales.

## Estado general: **FAIL** (FALLIDO)

Se detectaron errores estructurales confirmados que impedirían o comprometerían la importación.

_PASS significa que los archivos superaron los controles estructurales configurados. No significa que las traducciones sean lingüísticamente correctas ni que se haya realizado una importación real al CMS._

## Solicitud

Compará los archivos JSON de origen en inglés con sus versiones traducidas al español y al árabe. Investigá las reglas estructurales aplicables, identificá claves faltantes o inesperadas, diferencias de tipos de datos, problemas con placeholders y discrepancias en los saltos de línea. Determiná si los archivos cumplen los controles estructurales configurados para una posible importación al CMS y generá un informe de QA con evidencias.

## Resumen

- Errores: **6**
- Advertencias: **2**
- Información: **7**
- Pasos ejecutados: 8 de 12

- Idiomas de origen: en
- Idiomas de destino: ar, es
- Agentes que contribuyeron: analysis, research, supervisor, validation
- Controles ejecutados: claves_faltantes, claves_inesperadas, tipos_datos, placeholders, saltos_linea, metadatos

## Archivos analizados

- `/home/claude/l10n-qa-orchestrator/fixtures/source/clean.en.json` → `/home/claude/l10n-qa-orchestrator/fixtures/targets/clean.es.json` (en → es)
- `/home/claude/l10n-qa-orchestrator/fixtures/source/home.en.json` → `/home/claude/l10n-qa-orchestrator/fixtures/targets/home.ar.json` (en → ar)
- `/home/claude/l10n-qa-orchestrator/fixtures/source/home.en.json` → `/home/claude/l10n-qa-orchestrator/fixtures/targets/home.es.json` (en → es)

## Errores (6)


### F-007 · metadato_distinto · `_gpiInternalData-itemId`
- Archivo: `/home/claude/l10n-qa-orchestrator/fixtures/targets/home.ar.json` (idioma: ar)
- Regla: REG-MET-001 · Estado: confirmado
- Un metadato sensible a la importación difiere del origen y no tiene excepción.
- Evidencia: `{"valor_origen": "110d559f-dea5-42ea-9c1c-8a5df7e70ef9", "valor_destino": "110d559f-dea5-42ea-9c1c-8a5df7e70ef0"}`

### F-009 · clave_faltante · `page.description`
- Archivo: `/home/claude/l10n-qa-orchestrator/fixtures/targets/home.es.json` (idioma: es)
- Regla: REG-CLV-001 · Estado: confirmado
- La clave existe en el origen pero falta en el destino.
- Evidencia: `{"presente_en_origen": true, "presente_en_destino": false, "valor_origen": "Discover our products and services."}`

### F-010 · clave_inesperada · `page.promoBadge`
- Archivo: `/home/claude/l10n-qa-orchestrator/fixtures/targets/home.es.json` (idioma: es)
- Regla: REG-CLV-002 · Estado: confirmado
- La clave existe en el destino pero no en el origen.
- Evidencia: `{"presente_en_origen": false, "presente_en_destino": true, "valor_destino": "Nuevo"}`

### F-011 · tipo_distinto · `page.itemCount`
- Archivo: `/home/claude/l10n-qa-orchestrator/fixtures/targets/home.es.json` (idioma: es)
- Regla: REG-TIP-001 · Estado: confirmado
- El tipo de dato difiere: origen «number», destino «string».
- Evidencia: `{"tipo_origen": "number", "tipo_destino": "string", "valor_origen": 3, "valor_destino": "3"}`

### F-012 · placeholder_alterado · `page.banner.heading`
- Archivo: `/home/claude/l10n-qa-orchestrator/fixtures/targets/home.es.json` (idioma: es)
- Regla: REG-PH-001 · Estado: confirmado
- Los placeholders del destino no coinciden con los del origen.
- Evidencia: `{"faltantes_en_destino": ["{userName}"], "sobrantes_en_destino": ["{nombreUsuario}"], "placeholders_origen": ["{userName}"], "placeholders_destino": ["{nombreUsuario}"], "valor_origen": "Hello, {userName}!", "valor_destino": "¡Hola, {nombreUsuario}!"}`

### F-013 · salto_linea_tipo_distinto · `page.banner.legal`
- Archivo: `/home/claude/l10n-qa-orchestrator/fixtures/targets/home.es.json` (idioma: es)
- Regla: REG-NL-001 · Estado: confirmado
- Un salto de línea real del origen aparece como secuencia literal «\n» en el destino.
- Evidencia: `{"saltos_origen": {"reales": 1, "literales": 0}, "saltos_destino": {"reales": 0, "literales": 1}, "valor_origen": "Offer valid until {{endDate}}.\nTerms apply.", "valor_destino": "Oferta válida hasta {{endDate}}.\\nAplican condiciones."}`

## Advertencias (2)


### F-003 · valor_vacio · `page.links[1].label`
- Archivo: `/home/claude/l10n-qa-orchestrator/fixtures/targets/home.ar.json` (idioma: ar)
- Regla: REG-TIP-002 · Estado: pendiente
- El destino es una cadena vacía aunque el origen tiene texto.
- Evidencia: `{"valor_origen": "Support", "valor_destino": ""}`

### F-004 · salto_linea_cantidad_distinta · `footer.help`
- Archivo: `/home/claude/l10n-qa-orchestrator/fixtures/targets/home.ar.json` (idioma: ar)
- Regla: REG-NL-002 · Estado: pendiente
- La cantidad de saltos de línea difiere; requiere revisión.
- Evidencia: `{"saltos_origen": {"reales": 1, "literales": 0}, "saltos_destino": {"reales": 0, "literales": 0}, "valor_origen": "Need help?\nCall us at %s.", "valor_destino": "هل تحتاج إلى مساعدة؟ اتصل بنا على %s."}`

## Información (7)


### F-001 · metadato_diferencia_permitida · `_gpiInternalData-rootPath`
- Archivo: `/home/claude/l10n-qa-orchestrator/fixtures/targets/clean.es.json` (idioma: es)
- Regla: REG-EXC-003 · Estado: confirmado
- El metadato difiere del origen pero está permitido por la excepción EXC-002.
- Evidencia: `{"valor_origen": "/sitecore/content/acme/en/contact", "valor_destino": "/sitecore/content/acme/es/contact", "excepcion": "EXC-002"}`
- Excepción aplicada: EXC-002 — La ruta raíz del contenido incluye el segmento de idioma (p. ej. /en/ frente a /es/), por lo que difiere legítimamente.

### F-002 · diferencia_legitima · `$`
- Archivo: `/home/claude/l10n-qa-orchestrator/fixtures/targets/clean.es.json` (idioma: es)
- Regla: REG-DIF-001 · Estado: confirmado
- Diferencias legítimas que NO se consideran errores: 1 valor(es) idéntico(s) al origen y 2 con longitud notablemente distinta.
- Evidencia: `{"valores_identicos_al_origen": ["contact.brand"], "longitud_notablemente_distinta": ["contact.channels[0]", "contact.channels[1]"], "reglas": ["REG-DIF-001", "REG-DIF-002"]}`

### F-005 · metadato_diferencia_permitida · `_gpiInternalData-toolName`
- Archivo: `/home/claude/l10n-qa-orchestrator/fixtures/targets/home.ar.json` (idioma: ar)
- Regla: REG-EXC-003 · Estado: confirmado
- El metadato difiere del origen pero está permitido por la excepción EXC-001.
- Evidencia: `{"valor_origen": "Sitecore Content Editor", "valor_destino": "محرر محتوى Sitecore", "excepcion": "EXC-001"}`
- Excepción aplicada: EXC-001 — El nombre de la herramienta puede variar legítimamente por idioma o por el flujo de traducción; no afecta la importación.

### F-006 · metadato_diferencia_permitida · `_gpiInternalData-rootPath`
- Archivo: `/home/claude/l10n-qa-orchestrator/fixtures/targets/home.ar.json` (idioma: ar)
- Regla: REG-EXC-003 · Estado: confirmado
- El metadato difiere del origen pero está permitido por la excepción EXC-002.
- Evidencia: `{"valor_origen": "/sitecore/content/acme/en/home", "valor_destino": "/sitecore/content/acme/ar/home", "excepcion": "EXC-002"}`
- Excepción aplicada: EXC-002 — La ruta raíz del contenido incluye el segmento de idioma (p. ej. /en/ frente a /es/), por lo que difiere legítimamente.

### F-008 · diferencia_legitima · `$`
- Archivo: `/home/claude/l10n-qa-orchestrator/fixtures/targets/home.ar.json` (idioma: ar)
- Regla: REG-DIF-001 · Estado: confirmado
- Diferencias legítimas que NO se consideran errores: 2 valor(es) idéntico(s) al origen y 2 con longitud notablemente distinta.
- Evidencia: `{"valores_identicos_al_origen": ["page.brand", "footer.copyright"], "longitud_notablemente_distinta": ["page.description", "footer.help"], "reglas": ["REG-DIF-001", "REG-DIF-002"]}`

### F-014 · metadato_diferencia_permitida · `_gpiInternalData-rootPath`
- Archivo: `/home/claude/l10n-qa-orchestrator/fixtures/targets/home.es.json` (idioma: es)
- Regla: REG-EXC-003 · Estado: confirmado
- El metadato difiere del origen pero está permitido por la excepción EXC-002.
- Evidencia: `{"valor_origen": "/sitecore/content/acme/en/home", "valor_destino": "/sitecore/content/acme/es/home", "excepcion": "EXC-002"}`
- Excepción aplicada: EXC-002 — La ruta raíz del contenido incluye el segmento de idioma (p. ej. /en/ frente a /es/), por lo que difiere legítimamente.

### F-015 · diferencia_legitima · `$`
- Archivo: `/home/claude/l10n-qa-orchestrator/fixtures/targets/home.es.json` (idioma: es)
- Regla: REG-DIF-001 · Estado: confirmado
- Diferencias legítimas que NO se consideran errores: 2 valor(es) idéntico(s) al origen y 2 con longitud notablemente distinta.
- Evidencia: `{"valores_identicos_al_origen": ["page.brand", "footer.copyright"], "longitud_notablemente_distinta": ["page.title", "page.banner.cta"], "reglas": ["REG-DIF-001", "REG-DIF-002"]}`

## Reglas aplicadas

- **REG-CLV-001**: Toda clave del origen debe existir en el destino
- **REG-CLV-002**: No debe haber claves inesperadas en el destino
- **REG-DIF-001**: Un texto igual al origen no es un error por sí mismo
- **REG-EXC-001**: Excepción configurable: _gpiInternalData-toolName
- **REG-EXC-002**: Excepción configurable: _gpiInternalData-rootPath
- **REG-EXC-003**: Las excepciones se aplican por clave explícita y se documentan
- **REG-MET-001**: Los metadatos _gpiInternalData-* deben coincidir con el origen
- **REG-NL-001**: No se intercambian saltos de línea reales y secuencias \n literales
- **REG-NL-002**: Una cantidad distinta de saltos de línea requiere revisión
- **REG-PH-001**: Los placeholders deben conservarse sin alteraciones
- **REG-PH-002**: El orden de los placeholders puede variar
- **REG-TIP-001**: El tipo de dato JSON debe coincidir entre origen y destino
- **REG-TIP-002**: Un valor de texto vacío en el destino requiere revisión

## Excepciones aplicadas

- **EXC-002** en `_gpiInternalData-rootPath` (/home/claude/l10n-qa-orchestrator/fixtures/targets/clean.es.json): La ruta raíz del contenido incluye el segmento de idioma (p. ej. /en/ frente a /es/), por lo que difiere legítimamente.
- **EXC-001** en `_gpiInternalData-toolName` (/home/claude/l10n-qa-orchestrator/fixtures/targets/home.ar.json): El nombre de la herramienta puede variar legítimamente por idioma o por el flujo de traducción; no afecta la importación.
- **EXC-002** en `_gpiInternalData-rootPath` (/home/claude/l10n-qa-orchestrator/fixtures/targets/home.ar.json): La ruta raíz del contenido incluye el segmento de idioma (p. ej. /en/ frente a /es/), por lo que difiere legítimamente.
- **EXC-002** en `_gpiInternalData-rootPath` (/home/claude/l10n-qa-orchestrator/fixtures/targets/home.es.json): La ruta raíz del contenido incluye el segmento de idioma (p. ej. /en/ frente a /es/), por lo que difiere legítimamente.

## Validación final

- [x] Están presentes las contribuciones necesarias (investigación y análisis) — Investigación, análisis y supervisión contribuyeron.
- [x] Los hallazgos tienen una estructura válida — 15 hallazgo(s) con estructura válida.
- [x] Cada hallazgo tiene evidencia suficiente para reproducirlo — Todas las evidencias se reprodujeron releyendo los archivos.
- [x] Las traducciones y archivos originales permanecen intactos — 5 archivo(s) con huella SHA-256 idéntica antes y después.
- [x] No se superó el límite de ejecución — Ejecución 8 de 12 permitidas.
- [x] El informe refleja fielmente los hallazgos — El informe refleja fielmente los hallazgos producidos.

## Problemas pendientes

- Advertencia F-003 pendiente de revisión: /home/claude/l10n-qa-orchestrator/fixtures/targets/home.ar.json · page.links[1].label.
- Advertencia F-004 pendiente de revisión: /home/claude/l10n-qa-orchestrator/fixtures/targets/home.ar.json · footer.help.

## Recomendaciones

- Restaurar el valor del metadato sensible a la importación según el origen, o configurar una excepción justificada.
- Agregar en el archivo de destino las claves faltantes (el equipo de traducción debe aportar el texto).
- Revisar las claves inesperadas: eliminarlas del destino o confirmar si el origen debe actualizarse.
- Corregir el tipo de dato del destino para que coincida con el origen (p. ej. número, no cadena).
- Restaurar los placeholders del destino exactamente como en el origen (no se traducen).
- Restaurar el tipo de salto de línea del origen (real frente a secuencia literal \n).
- Revisar los valores vacíos del destino: pueden ser campos sin traducir o un fallo de exportación.
- Revisar si la diferencia en la cantidad de saltos de línea es intencional.

## Decisiones del Supervisor

1. → `research`: Falta evidencia: aún no se inspeccionaron los archivos ni se consultaron reglas.
3. → `research`: Las reglas recuperadas no cubren las categorías: tipos_datos, placeholders, metadatos, excepciones.
5. → `analysis`: La investigación es suficiente y falta el análisis estructural.
7. → `validation`: Investigación y análisis suficientes: se pasa a la validación final.


## 8. Estado de finalización

In [9]:
print("task_completed:", final["task_completed"])
print("next_agent:    ", final["next_agent"])
print("pasos:         ", f"{final['step_count']}/{final['max_steps']}")
print("agentes:       ", informe["agentes_contribuyentes"])

task_completed: True
next_agent:     END
pasos:          8/12
agentes:        ['analysis', 'research', 'supervisor', 'validation']


## 9. Diagrama del grafo compilado (Mermaid)

In [10]:
print(obtener_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	research(research)
	analysis(analysis)
	validation(validation)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	analysis -.-> supervisor;
	analysis -.-> validation;
	research -.-> supervisor;
	research -.-> validation;
	supervisor -. &nbsp;END&nbsp; .-> __end__;
	supervisor -.-> analysis;
	supervisor -.-> research;
	supervisor -.-> validation;
	validation -. &nbsp;END&nbsp; .-> __end__;
	validation -.-> supervisor;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 10. Otros escenarios

**Caso limpio**: un par de archivos sin errores estructurales devuelve `PASS`.
`PASS` significa solo que se superaron los controles estructurales configurados.

In [11]:
from app.cli import ejecutar_qa
limpio = ejecutar_qa(SOLICITUD_EJEMPLO, [str(RAIZ/"fixtures/source/clean.en.json")], [str(RAIZ/"fixtures/targets/clean.es.json")])
r = limpio["final_report"]
print(r["estado_general"], "-", r["estado_general_es"], "·", r["resumen"])
print(r["aclaracion"])

PASS - APROBADO · {'errores': 0, 'advertencias': 0, 'informacion': 2, 'total': 2}
PASS significa que los archivos superaron los controles estructurales configurados. No significa que las traducciones sean lingüísticamente correctas ni que se haya realizado una importación real al CMS.


**Límite de pasos**: con `max_pasos=5` el grafo no alcanza a analizar. Se cierra de forma controlada y
el resultado es `INCOMPLETE` (nunca se presenta como validado).

In [12]:
corto = ejecutar_qa(SOLICITUD_EJEMPLO, [ORIGEN], [DESTINO], max_pasos=5)
r = corto["final_report"]
print(r["estado_general"], "-", r["estado_general_es"], "· pasos:", corto["step_count"], "/", corto["max_steps"])
for p in r["problemas_pendientes"]:
    print("  pendiente:", p)

INCOMPLETE - INCOMPLETO · pasos: 5 / 5
  pendiente: El análisis no se completó: No se realizó el análisis (falta contribución del Agente de Análisis).
